# CPTAC External Validation: FiLM-Conditioned Attention-MIL

Runs the 5 TCGA-trained fold checkpoints on held-out CPTAC LUAD/LUSC cases.

**Expected Kaggle datasets attached to this notebook:**
1. `cptac-luad-part1`
2. `cptac-luad-part2`
3. `cptac-lusc`
4. `cptac-nsclc-metadata` — contains `cptac-nsclc-metadata.csv`
5. `nsclc-film-mil-checkpoints` — contains `fold0..4_best_model.pt` **and** `model_config.json`
   (make sure `model_config.json` is in this dataset — it's not listed as a separate
   dataset above, so it needs to live alongside the checkpoints)

This notebook does **not** need `train_film_mil.py`'s `FiLMDataset` — CPTAC file naming
breaks its fixed 12-character barcode slicing (TCGA-specific), and it only supports one
source directory per subtype. A dedicated `CPTACExternalDataset` below fixes both, and
concatenates all of a patient's slides into a single tile bag (one prediction per
patient, using all available tissue, without inflating the test-set N).

## Setup

In [ ]:
!pip install h5py --quiet

import os, json, re, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
# Dataset paths
LUAD_DIRS = [
    "/kaggle/input/datasets/lucashuitema/cptac-luad-part1",
    "/kaggle/input/datasets/lucashuitema/cptac-luad-part2",
]
LUSC_DIRS = [
    "/kaggle/input/datasets/lucashuitema/cptac-lusc",
]
METADATA_CSV = "/kaggle/input/datasets/lucashuitema/cptac-csv/cptac-nsclc-metadata.csv"
CHECKPOINT_DIR = "/kaggle/input/datasets/lucashuitema/nsclc-film-mil-checkpoints" # fold*.pt + model_config.json

OUTPUT_DIR = "/kaggle/working/cptac_external_validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for p in LUAD_DIRS + LUSC_DIRS + [METADATA_CSV, CHECKPOINT_DIR]:
    exists = Path(p).exists()
    print(f"{'OK ' if exists else 'MISSING'}  {p}")
    if not exists:
        print(f"Fix the missing path(s) before continuing")

## Model definition

This is based on train_film_mil.py so no import is required

In [ ]:
class AttentionMIL(nn.Module):
    """Additive attention pooling (Ilse et al., 2018)."""
    def __init__(self, feat_dim: int = 1536, hidden_dim: int = 256):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        self.feat_proj = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
        )

    def forward(self, features):
        assert features.ndim == 2, (
            f"AttentionMIL expects (N_tiles, feat_dim), got shape {tuple(features.shape)}"
        )
        projected = self.feat_proj(features)            # (N, 512)
        raw_attn  = self.attention(features)             # (N, 1)
        attn      = torch.softmax(raw_attn, dim=0)        # (N, 1)
        slide_embed = (attn * projected).sum(dim=0)       # (512,)
        return slide_embed, attn.squeeze(-1)


class FiLMLayer(nn.Module):
    """Feature-wise Linear Modulation (Perez et al., 2018)."""
    def __init__(self, embed_dim: int = 512, n_subtypes: int = 2):
        super().__init__()
        self.subtype_embed = nn.Embedding(n_subtypes, embed_dim)
        self.gamma = nn.Linear(embed_dim, embed_dim)
        self.beta  = nn.Linear(embed_dim, embed_dim)

    def forward(self, slide_embed, subtype_id):
        s     = self.subtype_embed(subtype_id)
        gamma = self.gamma(s)
        beta  = self.beta(s)
        return gamma * slide_embed + beta


class FiLMMILModel(nn.Module):
    """End-to-end FiLM-conditioned Attention-MIL model. Architecture must
    match train_film_mil.py EXACTLY (including the Sequential blocks in
    feat_proj/head) or load_state_dict will fail on key names/shapes."""
    def __init__(
        self,
        feat_dim:     int = 1536,
        embed_dim:    int = 512,
        clinical_dim: int = 2,
        n_clinical:   int = 64,
        n_genes:      int = 35,
        n_subtypes:   int = 2,
        dropout:      float = 0.25,
    ):
        super().__init__()
        self.attention_mil = AttentionMIL(feat_dim=feat_dim, hidden_dim=256)
        self.film = FiLMLayer(embed_dim=embed_dim, n_subtypes=n_subtypes)
        self.clinical_proj = nn.Sequential(
            nn.Linear(clinical_dim, n_clinical),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(embed_dim + n_clinical, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_genes),
        )

    def forward(self, features, clinical, subtype_id):
        slide_embed, attn_weights = self.attention_mil(features)
        slide_embed = self.film(slide_embed, subtype_id)
        clin_embed  = self.clinical_proj(clinical)
        combined = torch.cat([slide_embed, clin_embed], dim=-1)
        preds    = self.head(combined)
        return preds, attn_weights


print("Model classes defined.")

## CPTAC-specific Dataset

Regex-based case ID extraction (handles CPTAC's variable-length slide suffixes, unlike
the training script's fixed 12-char slice), multi-directory support per subtype, and
multi-slide concatenation into one tile bag per patient.

In [ ]:
CASE_ID_RE = re.compile(r"(C3[LN]-\d{5})", re.IGNORECASE)


class CPTACExternalDataset(Dataset):
    def __init__(self, df, feature_dirs, gene_cols, clinical_cols, n_tiles_cap=None):
        """
        feature_dirs: {"LUAD": [dir1, dir2, ...], "LUSC": [dir1, ...]}
        n_tiles_cap: optional cap on total tiles per patient (random subsample), if it gets too large for GPU memory.
        """
        self.gene_cols = gene_cols
        self.clinical_cols = clinical_cols
        self.n_tiles_cap = n_tiles_cap

        self.case_index = {}
        for subtype, dirs in feature_dirs.items():
            index = {}
            for d in dirs:
                for p in Path(d).glob("*.h5"):
                    m = CASE_ID_RE.search(p.stem)
                    if m is None:
                        continue
                    cid = m.group(1).upper()
                    index.setdefault(cid, []).append(p)
            self.case_index[subtype] = index
            n_slides = sum(len(v) for v in index.values())
            print(f"{subtype}: {len(index)} unique cases, {n_slides} total slides across {len(dirs)} source dir(s)")

        records = []
        n_ambiguous = 0
        for _, row in df.iterrows():
            sid = str(row["submitter_id"]).upper().strip()
            found = {st: idx[sid] for st, idx in self.case_index.items() if sid in idx}
            if len(found) == 0:
                continue
            if len(found) > 1:
                n_ambiguous += 1
                continue
            subtype, h5_paths = next(iter(found.items()))
            records.append({
                "sid": sid,
                "subtype": subtype,
                "h5_paths": h5_paths,
                "n_slides": len(h5_paths),
                "label": row[gene_cols].values.astype(np.float32),
                "clinical": row[clinical_cols].values.astype(np.float32),
            })
        self.records = records
        if n_ambiguous:
            print(f"\u26a0 skipped {n_ambiguous} case(s) found in both LUAD and LUSC dirs")
        n_luad = sum(1 for r in records if r["subtype"] == "LUAD")
        n_lusc = sum(1 for r in records if r["subtype"] == "LUSC")
        print(f"Dataset: {len(records)} matched patients (LUAD: {n_luad}, LUSC: {n_lusc})")
        multi = [r for r in records if r["n_slides"] > 1]
        if multi:
            print(f"{len(multi)} patient(s) have multiple slides concatenated "
                  f"into one bag (e.g. {multi[0]['sid']}: {multi[0]['n_slides']} slides)")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        chunks = []
        for p in rec["h5_paths"]:
            with h5py.File(p, "r") as f:
                feats = torch.tensor(f["features"][:], dtype=torch.float32)
            if feats.ndim == 3 and feats.shape[0] == 1:
                feats = feats.squeeze(0)
            elif feats.ndim != 2:
                raise ValueError(f"Unexpected shape {tuple(feats.shape)} in {p}")
            chunks.append(feats)
        features = torch.cat(chunks, dim=0)

        if self.n_tiles_cap is not None and features.shape[0] > self.n_tiles_cap:
            perm = torch.randperm(features.shape[0])[: self.n_tiles_cap]
            features = features[perm]

        label = torch.tensor(rec["label"], dtype=torch.float32)
        clinical = torch.tensor(rec["clinical"], dtype=torch.float32)
        subtype_id = torch.tensor(0 if rec["subtype"] == "LUAD" else 1, dtype=torch.long)
        return features, label, label.clone(), clinical, subtype_id, rec["sid"]


import h5py
print("CPTACExternalDataset defined.")

In [ ]:
# Load config + metadata, build dataset
with open(f"{CHECKPOINT_DIR}/model_config.json") as f:
    config = json.load(f)

gene_cols = config["gene_cols"]
clinical_cols = config["clinical_cols"]
raw_gene_cols = [c for c in gene_cols if c.endswith("_fpkm_uq")]
apm_genes = config["APM_GENES"]
tis_genes = config["TIS_GENES"]
N_FOLDS = config["training"]["n_folds"]

df = pd.read_csv(METADATA_CSV)
print(f"Loaded metadata: {len(df)} rows, {df['submitter_id'].nunique()} unique submitter_id")

missing_cols = [c for c in gene_cols + clinical_cols if c not in df.columns]
if missing_cols:
    print(f"\u26a0 CSV is missing expected columns: {missing_cols}")
    print("Check this against model_config.json's gene_cols/clinical_cols before continuing.")

test_ds = CPTACExternalDataset(
    df,
    feature_dirs={"LUAD": LUAD_DIRS, "LUSC": LUSC_DIRS},
    gene_cols=gene_cols,
    clinical_cols=clinical_cols,
    n_tiles_cap=None,  # set e.g. 20000 if hit GPU OOM is hit on multi-slide patients
)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

test_sids = [r["sid"] for r in test_ds.records]
test_subtypes = [r["subtype"] for r in test_ds.records]

## Run inference for each fold checkpoint

Same fixed test set for every fold (all CPTAC cases), matching how the TCGA held-out test set was evaluated: 5 independently-trained models, one shared test set.

In [ ]:
def run_inference(model, loader, device):
    model.eval()
    all_preds, all_labels, all_sids = [], [], []
    with torch.no_grad():
        for features, label, _, clinical, subtype_id, sid in loader:
            features   = features.squeeze(0).to(device)
            label      = label.squeeze(0).to(device)
            clinical   = clinical.squeeze(0).to(device)
            subtype_id = subtype_id.squeeze(0).to(device)

            pred, _ = model(features, clinical, subtype_id)
            all_preds.append(pred.cpu().numpy())
            all_labels.append(label.cpu().numpy())
            all_sids.append(sid[0])
    return np.array(all_preds), np.array(all_labels), all_sids


all_fold_preds = {}
all_fold_labels = None
sid_order = None

for fold in range(N_FOLDS):
    ckpt_path = Path(CHECKPOINT_DIR) / f"fold{fold}_best_model.pt"
    if not ckpt_path.exists():
        print(f"Fold {fold}: checkpoint not found at {ckpt_path}, skipping")
        continue

    model = FiLMMILModel(feat_dim=1536, n_genes=len(gene_cols)).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    preds, labels, sids = run_inference(model, test_loader, device)
    all_fold_preds[fold] = preds
    all_fold_labels = labels
    sid_order = sids
    print(f"Fold {fold}: inference done, preds shape {preds.shape}")

ensemble_preds = np.mean(list(all_fold_preds.values()), axis=0)
print(f"\u2713 Ensemble predictions computed from {len(all_fold_preds)} folds")

### Save raw predictions

In [ ]:
for fold, preds in all_fold_preds.items():
    np.savez(
        f"{OUTPUT_DIR}/fold{fold}_cptac_predictions.npz",
        preds=preds, labels=all_fold_labels,
        submitter_id=np.array(sid_order), subtype=np.array(test_subtypes),
        gene_cols=np.array(gene_cols),
    )

np.savez(
    f"{OUTPUT_DIR}/ensemble_cptac_predictions.npz",
    preds=ensemble_preds, labels=all_fold_labels,
    submitter_id=np.array(sid_order), subtype=np.array(test_subtypes),
    gene_cols=np.array(gene_cols),
)
print(f"Saved raw predictions to {OUTPUT_DIR}")

## Analysis

### Panel PCC / AUC — per fold and ensemble

Same panel-score definition used throughout (mean of z-scored panel genes), so this is directly comparable to the TCGA held-out test-set numbers.

In [ ]:
from sklearn.metrics import roc_auc_score

def panel_score(arr, gene_cols, panel_genes):
    cols_idx = [gene_cols.index(f"{g}_fpkm_uq") for g in panel_genes if f"{g}_fpkm_uq" in gene_cols]
    return arr[:, cols_idx].mean(axis=1)

def panel_pcc(preds, labels, gene_cols, panel_genes):
    x = panel_score(preds, gene_cols, panel_genes)
    y = panel_score(labels, gene_cols, panel_genes)
    return float(np.corrcoef(x, y)[0, 1])

def panel_auc(preds, labels, gene_cols, panel_genes):
    pred_score = panel_score(preds, gene_cols, panel_genes)
    label_score = panel_score(labels, gene_cols, panel_genes)
    thresh = np.percentile(label_score, 75)
    y_true = (label_score >= thresh).astype(int)
    if y_true.sum() in (0, len(y_true)):
        return float("nan")
    return float(roc_auc_score(y_true, pred_score))

results_summary = {}
for name, preds in {**{f"fold{f}": p for f, p in all_fold_preds.items()}, "ensemble": ensemble_preds}.items():
    results_summary[name] = {}
    for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
        pcc = panel_pcc(preds, all_fold_labels, gene_cols, genes)
        auc = panel_auc(preds, all_fold_labels, gene_cols, genes)
        results_summary[name][panel] = {"PCC": pcc, "AUC": auc}
        print(f"{name:10s} {panel}: PCC={pcc:.4f}  AUC={auc:.4f}")

with open(f"{OUTPUT_DIR}/cptac_results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)

### Bootstrap CIs + permutation p-values

Same approach as the TCGA test-set stats — resampling at the **patient** level (since each patient contributes exactly one row here regardless of how many slides went into their prediction).

In [ ]:
N_BOOT = 2000
N_PERM = 2000
rng = np.random.default_rng(0)

def bootstrap_ci(preds, labels, gene_list, n_boot=N_BOOT):
    n = preds.shape[0]
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals.append(panel_pcc(preds[idx], labels[idx], gene_cols, gene_list))
    vals = np.array(vals)
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def permutation_pvalue(preds, labels, gene_list, n_perm=N_PERM):
    observed = panel_pcc(preds, labels, gene_cols, gene_list)
    n = preds.shape[0]
    null_vals = np.empty(n_perm)
    for i in range(n_perm):
        perm_idx = rng.permutation(n)
        null_vals[i] = panel_pcc(preds, labels[perm_idx], gene_cols, gene_list)
    p = (np.sum(null_vals >= observed) + 1) / (n_perm + 1)
    return float(observed), float(p)

boot_stats = {}
for name, preds in {**{f"fold{f}": p for f, p in all_fold_preds.items()}, "ensemble": ensemble_preds}.items():
    boot_stats[name] = {}
    for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
        mean_pcc, lo, hi = bootstrap_ci(preds, all_fold_labels, genes)
        observed, p = permutation_pvalue(preds, all_fold_labels, genes)
        boot_stats[name][panel] = {"PCC": observed, "CI95_low": lo, "CI95_high": hi, "perm_p": p}
        print(f"{name:10s} {panel}: PCC={observed:.4f} 95%CI=[{lo:.4f},{hi:.4f}] p={p:.4f}")

with open(f"{OUTPUT_DIR}/cptac_bootstrap_stats.json", "w") as f:
    json.dump(boot_stats, f, indent=2)

### Subtype-stratified breakdown (LUAD vs LUSC)

In [ ]:
subtypes_arr = np.array(test_subtypes)
subtype_rows = []
for subtype in ["LUAD", "LUSC"]:
    mask = subtypes_arr == subtype
    n = mask.sum()
    for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
        pcc = panel_pcc(ensemble_preds[mask], all_fold_labels[mask], gene_cols, genes)
        subtype_rows.append({"subtype": subtype, "panel": panel, "n": int(n), "PCC": pcc})

subtype_df = pd.DataFrame(subtype_rows)
subtype_df.to_csv(f"{OUTPUT_DIR}/cptac_subtype_breakdown.csv", index=False)
print(subtype_df)

### Compare against TCGA held-out test-set numbers

The TCGA test-set PCCs (from the earlier `results.json` / `bootstrap_permutation_stats.json`) are filled in here for comparison.

In [ ]:
TCGA_TEST_PCC = {
    "APM": 0.5964123606681824,  # filled in from TCGA bootstrap_permutation_stats.json -> ensemble -> APM -> PCC
    "TIS": 0.7259935140609741,  # same, for TIS
}

print("Internal (TCGA) vs External (CPTAC) ensemble PCC:")
for panel in ["APM", "TIS"]:
    cptac_pcc = results_summary["ensemble"][panel]["PCC"]
    tcga_pcc = TCGA_TEST_PCC[panel]
    if tcga_pcc is not None:
        print(f"  {panel}: TCGA={tcga_pcc:.4f}  CPTAC={cptac_pcc:.4f}  "
              f"delta={cptac_pcc - tcga_pcc:+.4f}")
    else:
        print(f"{panel}: CPTAC={cptac_pcc:.4f}  (fill in TCGA_TEST_PCC above to compare)")

### Per-gene PCC for CPTAC (with FDR)

In [ ]:
from scipy.stats import pearsonr

try:
    from statsmodels.stats.multitest import multipletests
    def fdr_correct(pvals):
        return multipletests(pvals, method="fdr_bh")[1]
except ImportError:
    def fdr_correct(pvals):
        pvals = np.asarray(pvals); n = len(pvals)
        order = np.argsort(pvals)
        ranked = pvals[order] * n / (np.arange(n) + 1)
        ranked = np.minimum.accumulate(ranked[::-1])[::-1]
        out = np.empty(n); out[order] = np.clip(ranked, 0, 1)
        return out

gene_symbols = [c.replace("_fpkm_uq", "") for c in gene_cols if c.endswith("_fpkm_uq")]
gene_idx = [gene_cols.index(f"{g}_fpkm_uq") for g in gene_symbols]

rows = []
for g, i in zip(gene_symbols, gene_idx):
    r, p = pearsonr(ensemble_preds[:, i], all_fold_labels[:, i])
    panel = "APM" if g in apm_genes else ("TIS" if g in tis_genes else "other")
    rows.append({"gene": g, "PCC": r, "p": p, "panel": panel})

gene_df = pd.DataFrame(rows).sort_values("PCC", ascending=False)
gene_df["q"] = fdr_correct(gene_df["p"].values)
gene_df.to_csv(f"{OUTPUT_DIR}/cptac_gene_level_pcc.csv", index=False)
print(gene_df.to_string(index=False))

### Subtype-level bootstrap CIs

In [ ]:
subtype_ci_rows = []
for subtype in ["LUAD", "LUSC"]:
    mask = subtypes_arr == subtype
    for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
        mean_pcc, lo, hi = bootstrap_ci(ensemble_preds[mask], all_fold_labels[mask], genes)
        observed, p = permutation_pvalue(ensemble_preds[mask], all_fold_labels[mask], genes)
        subtype_ci_rows.append({
            "subtype": subtype, "panel": panel, "n": int(mask.sum()),
            "PCC": observed, "CI95_low": lo, "CI95_high": hi, "perm_p": p,
        })

subtype_ci_df = pd.DataFrame(subtype_ci_rows)
subtype_ci_df.to_csv(f"{OUTPUT_DIR}/cptac_subtype_breakdown_with_CI.csv", index=False)
print(subtype_ci_df.to_string(index=False))

### Patient-level manifest

In [ ]:
manifest = pd.DataFrame({
    "submitter_id": test_sids,
    "subtype": test_subtypes,
    "n_slides": [r["n_slides"] for r in test_ds.records],
})
manifest.to_csv(f"{OUTPUT_DIR}/cptac_patient_manifest.csv", index=False)
print(f"{len(manifest)} patients in final CPTAC test set "
      f"({(manifest.subtype=='LUAD').sum()} LUAD, {(manifest.subtype=='LUSC').sum()} LUSC)")
print(f"Multi-slide patients: {(manifest.n_slides > 1).sum()}")

### Self-document the run

In [ ]:
shutil.copy(f"{CHECKPOINT_DIR}/model_config.json", f"{OUTPUT_DIR}/model_config_used.json")
run_metadata = {
    "n_folds_loaded": len(all_fold_preds),
    "n_test_patients": len(test_ds),
    "tcga_vs_cptac_ensemble": {
        "APM": {"TCGA": TCGA_TEST_PCC["APM"], "CPTAC": results_summary["ensemble"]["APM"]["PCC"]},
        "TIS": {"TCGA": TCGA_TEST_PCC["TIS"], "CPTAC": results_summary["ensemble"]["TIS"]["PCC"]},
    },
}
with open(f"{OUTPUT_DIR}/run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2)
print("Saved model_config_used.json and run_metadata.json")